In [1]:
#Task 1: 3-day lag and 7-day rolling average features
import pandas as pd
import numpy as np

df = pd.read_csv("ahmedabad_temperature.csv", parse_dates=["date"], index_col="date")

df["lag3"] = df["temperature"].shift(3)
df["roll7"] = df["temperature"].rolling(window=7).mean()

print(df.head(10))

            temperature  lag3      roll7
date                                    
2021-01-01         21.1   NaN        NaN
2021-01-02         20.3   NaN        NaN
2021-01-03         19.7   NaN        NaN
2021-01-04         20.3  21.1        NaN
2021-01-05         22.4  20.3        NaN
2021-01-06         21.4  19.7        NaN
2021-01-07         19.7  20.3  20.700000
2021-01-08         19.1  22.4  20.414286
2021-01-09         22.4  21.4  20.714286
2021-01-10         21.3  19.7  20.942857


In [2]:
#Task 2: Date-time features from Flipkart sales data
fs = pd.read_csv("flipkart_sales.csv", parse_dates=["order_date"])

fs["day_of_week"] = fs["order_date"].dt.day_name()
fs["month"] = fs["order_date"].dt.month
fs["is_weekend"] = (fs["order_date"].dt.dayofweek >= 5).astype(int)

print(fs.head(10))

   order_id order_date  amount day_of_week  month  is_weekend
0      1189 2023-01-01     693      Sunday      1           1
1      1291 2023-01-01     453      Sunday      1           1
2      1141 2023-01-03    3769     Tuesday      1           0
3      1295 2023-01-04    1039   Wednesday      1           0
4      1039 2023-01-07    3322    Saturday      1           1
5      1209 2023-01-09    4362      Monday      1           0
6      1199 2023-01-10     679     Tuesday      1           0
7      1062 2023-01-12     576    Thursday      1           0
8      1116 2023-01-13    4397      Friday      1           0
9      1262 2023-01-14     282    Saturday      1           1


In [3]:
#Task 3: Random Forest to forecast the next day's temperature
from sklearn.ensemble import RandomForestRegressor

# target = temperature of the next day, features = lag3 and roll7 known today
df["target"] = df["temperature"].shift(-1)

last_row = df.iloc[[-1]]              # last day, its next day is not known yet
data = df.dropna()

split = int(len(data) * 0.8)          # time based split, no shuffling
train = data.iloc[:split]
test = data.iloc[split:]
feats = ["lag3", "roll7"]

rf = RandomForestRegressor(n_estimators=100, random_state=1)
rf.fit(train[feats], train["target"])

next_date = last_row.index[0] + pd.Timedelta(days=1)
pred_next = rf.predict(last_row[feats])[0]
print("Predicted temperature for", next_date.date(), ":", round(pred_next, 1))

Predicted temperature for 2024-01-01 : 23.8


In [4]:
#Task 4: MAE, RMSE and MAPE on the test set
from sklearn.metrics import mean_absolute_error, mean_squared_error

pred = rf.predict(test[feats])
actual = test["target"]

def mape(a, p):
    return np.mean(np.abs((a - p) / a)) * 100

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("MAPE:", round(mape(actual, pred), 2), "%")

# RMSE is the most sensitive to large errors, because it squares each error before averaging, so one big miss increases it a lot.

MAE: 1.676
RMSE: 2.063
MAPE: 5.45 %


In [5]:
#Task 5: SMAPE (Symmetric Mean Absolute Percentage Error)
def smape(a, p):
    a = np.array(a)
    p = np.array(p)
    return np.mean(2 * np.abs(a - p) / (np.abs(a) + np.abs(p))) * 100

print("SMAPE:", round(smape(actual, pred), 2), "%")

# SMAPE divides by the average of the actual and predicted values, so it stays between 0 and 200 percent
# and it is not blown up when the actual value is very small, which can happen with MAPE.

SMAPE: 5.39 %
